# Notebook 8: Experiment 4 — Multi-Stock Training (70/30)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on 3 stocks combined, predict on the remaining 1 stock.  
**Train/Test Split:** 70/30 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Scaler:** ProportionScaler (÷ 10,501)  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  

**Combinations:**
- Train TLKM+BBCA+ASII → Predict UNVR
- Train TLKM+BBCA+UNVR → Predict ASII
- Train TLKM+ASII+UNVR → Predict BBCA
- Train BBCA+ASII+UNVR → Predict TLKM


In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

import plotly.graph_objects as go
from plotly.subplots import make_subplots

set_seed()
set_ieee_style()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.7
RATIO_LABEL = '70_30'
EXP_LABEL = f'Exp4_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 4 - Multi-Stock Training (70/30)")


Experiment 4 - Multi-Stock Training (70/30)


In [3]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


## Define Training Combinations

In [4]:
# ============================================================
# MULTI-STOCK COMBINATIONS
# ============================================================
# Each entry: (training_stocks, target_stock)
combinations = []
for target in STOCKS:
    train_stocks = [s for s in STOCKS if s != target]
    combinations.append((train_stocks, target))
    print(f"  Train: {', '.join(train_stocks)} -> Predict: {target}")


  Train: BBCA, ASII, UNVR -> Predict: TLKM
  Train: TLKM, ASII, UNVR -> Predict: BBCA
  Train: TLKM, BBCA, UNVR -> Predict: ASII
  Train: TLKM, BBCA, ASII -> Predict: UNVR


## Run All Multi-Stock Experiments

In [5]:
# ============================================================
# EXPERIMENT 4: Multi-stock training
# ============================================================
all_results = []
all_predictions = {}

for train_stocks, target_stock in combinations:
    train_label = '+'.join(train_stocks)
    pair_key = (train_label, target_stock)
    
    print(f"\n{'#'*60}")
    print(f"# TRAIN: {train_label} -> TARGET: {target_stock}")
    print(f"{'#'*60}")
    
    # Prepare multi-stock data
    train_dfs = [daily_data[s] for s in train_stocks]
    test_df = daily_data[target_stock]
    
    X_train, y_train, X_test, y_test, test_dates = prepare_multi_stock_data(
        train_dfs, test_df,
        train_ratio=TRAIN_RATIO, lookback=LOOKBACK
    )
    print(f"  X_train (combined): {X_train.shape}, X_test: {X_test.shape}")
    
    all_predictions[pair_key] = {}
    
    for model_type in MODEL_TYPES:
        exp_name = f'{EXP_LABEL}_train_{train_label}_target_{target_stock}'
        
        y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(
            model_type=model_type,
            X_train=X_train, y_train=y_train,
            X_test=X_test, y_test=y_test,
            experiment_name=exp_name,
            save_dir=f'models/{EXP_LABEL}',
            epochs=EPOCHS, batch_size=BATCH_SIZE
        )
        
        result = {
            'Train_Stocks': train_label,
            'Target_Stock': target_stock,
            'Model': model_type,
            **metrics
        }
        all_results.append(result)
        all_predictions[pair_key][model_type] = (y_true_inv, y_pred_inv, test_dates)
        
        plot_actual_vs_predicted(
            test_dates, y_true_inv, y_pred_inv,
            model_type, f'Train_{train_label}_Target_{target_stock}',
            EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )

print("\n\nAll Experiment 4 (70/30) training complete!")



############################################################
# TRAIN: BBCA+ASII+UNVR -> TARGET: TLKM
############################################################
  X_train (combined): (10831, 60, 1), X_test: (1573, 60, 1)

Training BiLSTM for: Exp4_70_30_train_BBCA+ASII+UNVR_target_TLKM
  Train samples: 10831, Test samples: 1573
Epoch 1/100
153/153 [==============================] - ETA: 0s - loss: 0.0023
Epoch 1: val_loss improved from inf to 0.00029, saving model to models/Exp4_70_30\Exp4_70_30_train_BBCA+ASII+UNVR_target_TLKM_BiLSTM_best.keras
153/153 [==============================] - 18s 41ms/step - loss: 0.0023 - val_loss: 2.9070e-04
Epoch 2/100
153/153 [==============================] - ETA: 0s - loss: 5.8780e-04
Epoch 2: val_loss improved from 0.00029 to 0.00025, saving model to models/Exp4_70_30\Exp4_70_30_train_BBCA+ASII+UNVR_target_TLKM_BiLSTM_best.keras
153/153 [==============================] - 5s 30ms/step - loss: 5.8780e-04 - val_loss: 2.4874e-04
Epoch 3/100
153/153 [==

## Results Summary

In [6]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 4 - Multi-Stock Training (70/30)")

results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 4 - Multi-Stock Training (70/30)
  Train_Stocks Target_Stock  Model        MSE     RMSE      MAE  MAPE (%)       R2  Training_Time_s  Epochs_Run
BBCA+ASII+UNVR         TLKM BiLSTM  5398.6223  73.4753  56.0559    1.9787 0.974368            508.7         100
BBCA+ASII+UNVR         TLKM  BiGRU  7203.2847  84.8722  65.9010    2.3415 0.965800            611.4         100
BBCA+ASII+UNVR         TLKM   LSTM  7135.7013  84.4731  69.1244    2.3335 0.966121            363.7         100
BBCA+ASII+UNVR         TLKM    GRU  9622.1029  98.0923  83.7195    2.8105 0.954316            348.1         100
TLKM+ASII+UNVR         BBCA BiLSTM 54980.4668 234.4791 186.7951    2.4586 0.977989            632.3         100
TLKM+ASII+UNVR         BBCA  BiGRU 16709.8881 129.2667  97.8676    1.4056 0.993310            640.5         100
TLKM+ASII+UNVR         BBCA   LSTM 56615.0911 237.9393 205.1212    2.7770 0.977334            371.2         100
TLKM+ASII+UNVR         BBCA    GRU 62275.0493 249.5497 22

## Visualizations

In [ ]:
# ============================================================
# INTERACTIVE RESULTS VISUALIZATIONS
# ============================================================

print("Generating interactive results visualizations...\n")

# 1. Interactive Metrics Comparison
print("1. Generating Metrics Comparison Chart...")
fig1, html1 = create_interactive_metrics_comparison(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}',
    metrics=['RMSE', 'MAE', 'R2']
)
print(f"   ✓ Saved: {html1}")
fig1.show()

print()

# 2. Model Radar Chart
print("2. Generating Model Radar Chart...")
fig2, html2 = create_interactive_model_radar_chart(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html2}")
fig2.show()

print()

# 3. Interactive Dashboard with Subplots
print("3. Generating Results Dashboard...")
fig3 = go.Figure()

# RMSE by model
rmse_by_model = results_df.groupby('Model')['RMSE'].mean().sort_values()
fig3.add_trace(go.Bar(
    x=rmse_by_model.index,
    y=rmse_by_model.values,
    name='RMSE',
    marker_color='#0072B2',
    text=np.round(rmse_by_model.values, 4),
    textposition='outside',
    hovertemplate='Model: %{x}<br>Avg RMSE: %{y:.4f}<extra></extra>'
))

fig3.update_layout(
    title=f"<b>{EXP_LABEL} - Average Metrics by Model</b>",
    xaxis_title="Model",
    yaxis_title="RMSE",
    height=600,
    template='plotly_white',
    font=dict(size=12),
    showlegend=False
)

html3 = f'figures/{EXP_LABEL}/{EXP_LABEL}_metrics_dashboard.html'
fig3.write_html(html3)
print(f"   ✓ Saved: {html3}")
fig3.show()

print("\n✓ All interactive results visualizations generated successfully!")

## Interactive Results Visualizations

In [ ]:
# ============================================================
# COMPARISON PLOTS
# ============================================================
for (train_label, target_stock), preds_dict in all_predictions.items():
    y_true = preds_dict[MODEL_TYPES[0]][0]
    dates = preds_dict[MODEL_TYPES[0]][2]
    preds = {mt: preds_dict[mt][1] for mt in MODEL_TYPES if mt in preds_dict}
    
    plot_all_models_comparison(
        dates, y_true, preds,
        f'Train_{train_label}_Target_{target_stock}',
        EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
    )

# Metrics bar chart
for metric in ['RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_comparison_bar(
        results_df, metric, EXP_LABEL,
        group_col='Target_Stock', save_dir=f'figures/{EXP_LABEL}'
    )

print("All visualizations saved!")


In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("  BEST MODEL PER TARGET STOCK (by RMSE)")
print("="*70)
for target in STOCKS:
    target_data = results_df[results_df['Target_Stock'] == target]
    if target_data.empty:
        continue
    best_idx = target_data['RMSE'].idxmin()
    best = target_data.loc[best_idx]
    print(f"  Target {target}: Trained on {best['Train_Stocks']} + {best['Model']} "
          f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")
